# FDS 메인 분류 모델 — LightGBM 재학습 (공통 분할본 v2)

팀 project · 개발팀(PL) · 기존 `FDS_ML_모델개발_Colab.ipynb` 파이프라인 재사용판

**기존 대비 바뀐 것**

1. 데이터: 파일 내부 분할 → **공통 분할본 fds_split_v2** (train/valid/test.parquet) 직접 로드
2. 하이퍼파라미터: Optuna 재탐색 없이 **기존 최적값 고정** → 총 소요 약 1시간 10분 (5시드)
3. 피처: 팀 공지 공통 제외 컬럼 적용 (신규 ID·절대시점 컬럼 7개 추가 제외)
4. **G_* 그래프 피처 선택 병합**: `graph_feats_{split}.parquet`이 있으면 자동 병합 — 없이 1회(베이스라인), 받고 1회(+G) 돌리면 그 차이가 ablation
5. 평가: threshold·캘리브레이션은 valid에서 확정, 전체/신규 이용자 구간 두 눈금. test는 `RUN_TEST=True`일 때만 1회

> CPU 런타임으로 충분 (지난번과 동일). 예상: 시드당 ~12분 × 5 + 전처리 ~10분.


## 0. Config

In [ ]:
DATA_DIR = "/content/data"
TARGET   = "INJSTUSE_AT"
SEED     = 42
SEEDS    = [42, 7, 123, 2024, 99]        # 급하면 [42]로 스모크 테스트(12분) 후 5시드
RUN_TEST = False                          # test 최종 1회 평가 시에만 True

# ---- 기존 파이프라인 상수 (그대로) ----
DROP_COLS = ['CONFM_NO','DELNG_SN','BSNS_SE_ID','TRGTER_LTR_DPLCT_AT','INORG_HSPTLZ_AT',
             'DELNG_MONTH','DELNG_QUARTER',
             # 팀 공지 추가 제외 (신규 파일 컬럼)
             'row_id','TRGTER_ID','PRHNF_ID','DELNG_DT','DELNG_DE',
             'SVC_BEGIN_DT','SVC_END_DT','PREDELNG_DE']
AMOUNT_COLS = ['GVSPMN_AM','SFALMT_AM','DDCT_PONT_SM_AM','REMNDR_PONT_AM','CREAT_PONT_AM',
               'CREAT_SFALMT_AM','CREAT_GVSPMN_AM','HLTHINS_CALC_AM','F_AMT_PER_MIN']
UNIT_COL = 'VOUCH_UNTPC'

# ---- 고정 하이퍼파라미터 ----
# 기본은 기존 수기 튜닝값. Optuna 최적값이 확인되는 항목은 그 값으로 덮어씀.
# ⚠️ reg_lambda·cat_smooth·cat_l2·min_data_per_group·scale_pos_weight 의 Optuna 최적값은
#    ml_report.json(best_params)에서 확인해 아래에 직접 기입할 것. 미기입 시 수기 기본값 사용.
BEST_PARAMS = dict(
    objective='binary',
    # Optuna 30회 최적 (수행보고서 부록 B)
    max_depth=11, num_leaves=112, learning_rate=0.031,
    min_child_samples=52, colsample_bytree=0.5076,
    # 수기 기본값 — ml_report.json 값으로 교체 권장
    reg_lambda=5.0, cat_smooth=100, cat_l2=25.0, min_data_per_group=200,
    scale_pos_weight=1.0,
    # 고정 설정
    subsample=0.8, subsample_freq=1, max_cat_to_onehot=8, max_cat_threshold=64,
    n_estimators=3000, verbose=-1,
)
ES_ROUNDS = 100

## 1. 준비 + 파일 업로드

분할본 3개(train/valid/test.parquet 또는 fds_split_v2.zip)는 필수. `graph_feats_*.parquet`·`ml_report.json`은 있으면 같이 올리면 자동 반영.

In [ ]:
import os, gc, json, time, pickle, zipfile, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip -q install lightgbm')
    import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score, precision_recall_curve)
np.random.seed(SEED)
print('lightgbm', lgb.__version__, '| pandas', pd.__version__)

os.makedirs(DATA_DIR, exist_ok=True)
PATHS = {s: f"{DATA_DIR}/{s}.parquet" for s in ("train","valid","test")}

def _missing(): return [p for p in PATHS.values() if not os.path.exists(p)]
if _missing():
    from google.colab import files as colab_files
    print("train/valid/test.parquet (또는 fds_split_v2.zip)를 업로드하세요. "
          "graph_feats_*.parquet, ml_report.json 도 있으면 같이.")
    for name in colab_files.upload():
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z: z.extractall(DATA_DIR)
            os.remove(name)
        else:
            os.replace(name, os.path.join(DATA_DIR, os.path.basename(name)))
assert not _missing(), f"없는 파일: {_missing()}"
for p in PATHS.values(): print(f"✅ {p} ({os.path.getsize(p)/1e6:.1f} MB)")

print('\n현재 데이터 폴더:', sorted(os.listdir(DATA_DIR)))

# ml_report.json이 있으면 Optuna best_params 자동 반영
_rp = os.path.join(DATA_DIR, 'ml_report.json')
if os.path.exists(_rp):
    try:
        _best = json.load(open(_rp)).get('best_params', {})
        for k, v in _best.items():
            if k in BEST_PARAMS: BEST_PARAMS[k] = v
        print('ml_report.json best_params 반영:', _best)
    except Exception as e:
        print('ml_report.json 파싱 실패 — 수기값 사용:', e)

## 1-b. (선택) 그래프 피처·ml_report 추가 업로드

**plus_graph 실행 때만 이 셀을 실행**해서 관계 피처 담당자 파일 3개를 올린다:
`graph_feats_train_recommended.parquet` / `graph_feats_valid_recommended.parquet` / `graph_feats_test_recommended.parquet`
(ml_report.json도 있으면 같이. baseline만 돌릴 거면 이 셀은 건너뛰기)

In [ ]:
from google.colab import files as colab_files
print("graph_feats_*_recommended.parquet 3개 (+ml_report.json)를 선택하세요.")
for name in colab_files.upload():
    os.replace(name, os.path.join(DATA_DIR, os.path.basename(name)))
print('현재 데이터 폴더:', sorted(os.listdir(DATA_DIR)))

# ml_report.json을 이 시점에 올렸으면 best_params 재반영
_rp = os.path.join(DATA_DIR, 'ml_report.json')
if os.path.exists(_rp):
    try:
        _best = json.load(open(_rp)).get('best_params', {})
        for k, v in _best.items():
            if k in BEST_PARAMS: BEST_PARAMS[k] = v
        print('ml_report.json best_params 반영:', _best)
    except Exception as e:
        print('ml_report.json 파싱 실패:', e)

## 2. 로드 + 전처리 — 기존 파이프라인 함수 그대로 (범주 레벨은 train 기준 고정)

In [ ]:
def read_parquet(path):
    df = pd.read_parquet(path)
    for c in df.columns:
        dt = str(df[c].dtype)
        if dt in ('string','large_string','object'):
            df[c] = df[c].astype(str).astype('category')
        elif dt.startswith(('Int','Float','boolean')):
            df[c] = df[c].astype('float64')
    return df

def upper_categories(df):
    for c in [c for c in df.columns if str(df[c].dtype)=='category']:
        s = df[c]
        mp = {x: str(x).strip().upper() for x in s.cat.categories}
        lv = sorted(set(mp.values()))
        pos = {v: i for i, v in enumerate(lv)}
        lut = np.array([pos[mp[x]] for x in s.cat.categories])
        cd = s.cat.codes.to_numpy()
        df[c] = pd.Categorical.from_codes(np.where(cd<0,-1,lut[np.clip(cd,0,None)]), categories=lv)
    return df

def add_relative(df):
    if UNIT_COL not in df.columns: return df
    unit = pd.to_numeric(df[UNIT_COL], errors='coerce').replace(0, np.nan)
    for c in AMOUNT_COLS:
        if c in df.columns:
            df[f'REL_{c}'] = (pd.to_numeric(df[c], errors='coerce')/unit).astype('float32')
    return df

def load_split(name):
    raw = pd.read_parquet(PATHS[name])
    row_id = raw['row_id'].to_numpy()
    trgter = raw['TRGTER_ID'].astype(str).to_numpy()
    y = pd.to_numeric(raw[TARGET]).astype(int).to_numpy()
    df = read_parquet(PATHS[name])
    df = upper_categories(df)
    X = df[[c for c in df.columns if c not in set(DROP_COLS)|{TARGET}]].copy()
    del df, raw; gc.collect()
    for c in X.columns:
        if str(X[c].dtype) != 'category':
            X[c] = pd.to_numeric(X[c], errors='coerce').astype('float32')
    X = add_relative(X)
    return X, y, row_id, trgter

Xtr, y_tr, rid_tr, tid_tr = load_split('train')
Xva, y_va, rid_va, tid_va = load_split('valid')
Xte, y_te, rid_te, tid_te = load_split('test')

# 범주 레벨을 train 기준으로 고정 (valid/test의 미등장 값 → NaN, LGBM이 결측으로 처리)
CATS = [c for c in Xtr.columns if str(Xtr[c].dtype)=='category']
for c in CATS:
    lv = Xtr[c].cat.categories
    Xva[c] = Xva[c].cat.set_categories(lv)
    Xte[c] = Xte[c].cat.set_categories(lv)

# 신규 이용자 구간 플래그 (팀 공지 기준)
_tr_ids = set(tid_tr)
new_va = ~np.isin(tid_va, list(_tr_ids))
new_te = ~np.isin(tid_te, list(_tr_ids))

for nm, X, y in [('train',Xtr,y_tr),('valid',Xva,y_va),('test',Xte,y_te)]:
    print(f"{nm:6s} {X.shape} 양성률 {y.mean():.4f}")
print(f"valid 신규 이용자 구간 {new_va.sum():,}행({new_va.mean():.1%}) 양성률 {y_va[new_va].mean():.4f}")
print(f"test  신규 이용자 구간 {new_te.sum():,}행({new_te.mean():.1%}) 양성률 {y_te[new_te].mean():.4f}")
assert len(Xtr)==395733 and len(Xva)==67263 and len(Xte)==67227, '공통 분할본(fds_split_v2)이 아님'

## 3. G_* 그래프 피처 선택 병합 — 파일 있으면 자동, 없으면 베이스라인

In [ ]:
# 권고 5피처 파일(graph_feats_*_recommended.parquet) 우선, 없으면 전체 25피처 파일
def _gpath(s):
    for p in (f"{DATA_DIR}/graph_feats_{s}_recommended.parquet",
              f"{DATA_DIR}/graph_feats_{s}.parquet"):
        if os.path.exists(p): return p
    return None

USE_GRAPH = all(_gpath(s) for s in ('train','valid','test'))
RUN_TAG = 'plus_graph' if USE_GRAPH else 'baseline'

if USE_GRAPH:
    print('사용 파일:', [_gpath(s).split('/')[-1] for s in ('train','valid','test')])
    assert all('recommended' in _gpath(s) for s in ('train','valid','test')) or            all('recommended' not in _gpath(s) for s in ('train','valid','test')),            '권고본과 전체본이 섞임 — 세 split 모두 같은 종류로 통일'
    for nm, X, rid in [('train',Xtr,rid_tr),('valid',Xva,rid_va),('test',Xte,rid_te)]:
        g = pd.read_parquet(_gpath(nm))
        gcols = [c for c in g.columns if c != 'row_id']
        m = pd.DataFrame({'row_id': rid}).merge(g, on='row_id', how='left')
        assert len(m) == len(X), f'{nm}: graph_feats 행 수 불일치'
        _na = m[gcols].isna().mean().max()
        if _na > 0: print(f'  {nm}: G_* 결측 최대 {_na:.2%} → 0 대치 (row_id 매칭 확인 권장)')
        for c in gcols:
            X[c] = pd.to_numeric(m[c], errors='coerce').fillna(0).astype('float32').values
    print(f"G_* 피처 {len(gcols)}개 병합:", gcols)
else:
    print("graph_feats_* 없음 → 베이스라인(G_* 미포함)으로 진행")

FEATURES = list(Xtr.columns)
print(f"[{RUN_TAG}] 최종 피처 {len(FEATURES)}개 (범주형 {len(CATS)})")

## 4. 학습 — 고정 하이퍼파라미터 · 5시드 앙상블 (Optuna 재탐색 없음)

In [ ]:
def fit(seed):
    p = dict(BEST_PARAMS); p.update(random_state=seed, n_jobs=os.cpu_count() or 2)
    m = lgb.LGBMClassifier(**p)
    m.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], eval_metric='average_precision',
          categorical_feature=CATS,
          callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
    return m

models, t0 = [], time.time()
for sd in SEEDS:
    t1 = time.time(); m = fit(sd); models.append(m)
    p_va = m.predict_proba(Xva)[:, 1]
    print(f"seed {sd:5d} | trees {m.best_iteration_} | valid PR-AUC "
          f"{average_precision_score(y_va, p_va):.4f} | {time.time()-t1:.0f}초")
print(f"\n총 학습 시간 {time.time()-t0:.0f}초 ({len(SEEDS)}시드)")

ens_va = np.mean([m.predict_proba(Xva)[:, 1] for m in models], axis=0)

## 5. 캘리브레이션 + threshold (valid에서 확정) → 두 눈금 평가

In [ ]:
iso = IsotonicRegression(out_of_bounds='clip').fit(ens_va, y_va)
cal_va = iso.transform(ens_va)

def best_f1_threshold(y_true, prob):
    pr, rc, th = precision_recall_curve(y_true, prob)
    d = pr[:-1] + rc[:-1]
    f1 = np.where(d>0, 2*pr[:-1]*rc[:-1]/np.where(d>0,d,1), 0.0)
    i = int(np.nanargmax(f1)); return float(th[i]), float(f1[i])

THR, _ = best_f1_threshold(y_va, cal_va)

def report(y_true, prob, thr, name, ks=(1000,3000,5000)):
    pred = (prob >= thr).astype(int)
    ap = average_precision_score(y_true, prob)
    line = (f"[{name}] n={len(y_true):,} 양성률 {y_true.mean():.4f} | "
            f"F1={f1_score(y_true, pred):.4f} "
            f"P={precision_score(y_true, pred, zero_division=0):.4f} "
            f"R={recall_score(y_true, pred):.4f} | PR-AUC={ap:.4f} "
            f"(lift {ap/max(y_true.mean(),1e-9):.2f}x) ROC-AUC={roc_auc_score(y_true, prob):.4f}")
    pk = []
    for k in ks:
        k = min(k, len(y_true)); top = np.argsort(-prob)[:k]
        pk.append(f"P@{k}={y_true[top].mean():.3f}")
    print(line); print('   ' + '  '.join(pk))

print(f"threshold = {THR:.4f} (valid 전체 F1 최대점)\n")
report(y_va, cal_va, THR, f"{RUN_TAG} | valid 전체 (참고)")
report(y_va[new_va], cal_va[new_va], THR, f"{RUN_TAG} | valid 신규 이용자 구간 (판정 기준)")

# ablation 대조용 결과 누적 저장
row = {'run': RUN_TAG, 'n_features': len(FEATURES), 'seeds': len(SEEDS), 'threshold': THR,
       'valid_full_F1': f1_score(y_va, (cal_va>=THR).astype(int)),
       'valid_new_F1': f1_score(y_va[new_va], (cal_va[new_va]>=THR).astype(int)),
       'valid_new_PRAUC': average_precision_score(y_va[new_va], cal_va[new_va])}
_ab = '/content/ablation_results.csv'
pd.DataFrame([row]).to_csv(_ab, mode='a', header=not os.path.exists(_ab), index=False)
print('\nablation_results.csv 누적 저장 →', pd.read_csv(_ab).to_string(index=False))

## 6. 산출물 저장 — ml_scores CSV + 모델 pkl

In [ ]:
OUT = f"/content/ml_out_{RUN_TAG}"; os.makedirs(OUT, exist_ok=True)

pd.DataFrame({'row_id': rid_va, 'y': y_va, 'score': cal_va}
             ).to_csv(f"{OUT}/ml_scores_valid.csv", index=False)

with open(f"{OUT}/ml_model_v2.pkl", 'wb') as f:
    pickle.dump({'models': models, 'calibrator': iso, 'threshold': THR,
                 'features': FEATURES, 'cats': CATS,
                 'cat_levels': {c: list(Xtr[c].cat.categories) for c in CATS},
                 'drop_cols': DROP_COLS, 'params': BEST_PARAMS, 'seeds': SEEDS,
                 'run_tag': RUN_TAG, 'use_graph': USE_GRAPH,
                 'lgb_version': lgb.__version__}, f)
print('저장:', sorted(os.listdir(OUT)))

## 7. test 최종 평가 — **확정 후 1회만.** `RUN_TEST=True`로 변경 후 실행

In [ ]:
if not RUN_TEST:
    print("RUN_TEST=False — test 미개봉 유지.")
else:
    ens_te = np.mean([m.predict_proba(Xte)[:, 1] for m in models], axis=0)
    cal_te = iso.transform(ens_te)
    pd.DataFrame({'row_id': rid_te, 'y': y_te, 'score': cal_te}
                 ).to_csv(f"{OUT}/ml_scores_test.csv", index=False)
    report(y_te, cal_te, THR, f"{RUN_TAG} | test 전체 (참고)")
    report(y_te[new_te], cal_te[new_te], THR, f"{RUN_TAG} | test 신규 이용자 구간 (판정 기준)")
    print("ml_scores_test.csv 저장 완료")

## 8. zip 다운로드

In [ ]:
zpath = f"/content/ml_out_{RUN_TAG}.zip"
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), fn)
    if os.path.exists('/content/ablation_results.csv'):
        z.write('/content/ablation_results.csv', 'ablation_results.csv')
print(f"{zpath} ({os.path.getsize(zpath)/1e6:.1f} MB)")
from google.colab import files as colab_files
colab_files.download(zpath)

## 사용 순서

1. **지금**: graph_feats 없이 실행 → `baseline` 결과 (약 1시간 10분, 스모크는 SEEDS=[42]로 12분)
2. **관계 피처 담당자 파일 도착 후**: `graph_feats_{train,valid,test}.parquet`을 `/content/data/`에 추가하고 **런타임 재시작 후 전체 재실행** → `plus_graph` 결과
3. `ablation_results.csv`의 `valid_new_F1` 차이 = G_* 기여도 (판정은 신규 이용자 구간 기준)
4. 최종 구성 확정 후 `RUN_TEST=True`로 test 1회 → `ml_scores_test.csv` 결합용 제출

주의: 기존 보고서 수치(F1 0.7771)는 구버전 데이터·준식별자 눈금 기준이라 직접 비교하지 말 것. 이번 수치는 새 데이터 + 신규 이용자 구간 눈금.